*Design notebook* — the borough cost aggregations and wall-type analysis here are implemented in production in `dbt/models/marts/`. This notebook documents the design rationale, improvement-type breakdowns, and cost scenario methodology that informed those models.

Run `dbt run --profiles-dir .` from the `dbt/` directory for the production path.

In [1]:
import os, glob
os.environ['JAVA_HOME'] = '/opt/homebrew/opt/openjdk@17'
os.environ['PYSPARK_PYTHON'] = '/Users/user/Documents/repos/.venv/bin/python'
os.environ['PYSPARK_DRIVER_PYTHON'] = '/Users/user/Documents/repos/.venv/bin/python'
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, count, countDistinct, avg, sum as spark_sum, 
    trim, when, lit, desc, udf, regexp_replace, round as spark_round
)
from pyspark.sql.types import FloatType
from pyspark.sql.window import Window
from pyspark.sql.functions import rank

spark = SparkSession.builder \
    .master('local[*]') \
    .appName('recommendations') \
    .config('spark.driver.memory', '24g') \
    .config('spark.sql.shuffle.partitions', '10') \
    .config('spark.sql.adaptive.enabled', 'true') \
    .config('spark.sql.adaptive.coalescePartitions.enabled', 'true') \
    .config('spark.driver.maxResultSize', '4g') \
    .getOrCreate()
spark.sparkContext.setLogLevel('ERROR')
print('Spark ready')

Spark ready


## 1. Load data and join to London social rented certificates

In [2]:
BRONZE = '../data/bronze/epc_raw'
SILVER = '../data/silver/epc'
GOLD   = '../data/gold'

rec_files  = glob.glob(f'{BRONZE}/recommendations-*.csv')
print(f'Recommendation files: {len(rec_files)}')

# Load certs from silver Parquet (already filtered to London social rented by nb03)
certs = (
    spark.read.parquet(SILVER)
    .select(
        col('certificate_id').alias('certificate_number'),
        col('borough'),
        col('epc_rating'),
        col('epc_score'),
        col('construction_age_band'),
        col('property_type')
    )
)

recs = spark.read.csv(rec_files, header=True, inferSchema=False) \
    .select(
        col('certificate_number'),
        col('improvement_item').cast('int').alias('priority'),
        col('improvement_id'),
        col('improvement_summary_text').alias('improvement_type'),
        col('indicative_cost')
    )

print(f'London social certs: {certs.count():,}')
print(f'Total recommendations rows: {recs.count():,}')

Recommendation files: 15
London social certs: 491,869
Total recommendations rows: 87,390,373


In [3]:
# Join recommendations to London social rented certs
joined = recs.join(certs, on='certificate_number', how='inner')
print(f'Joined rows (recommendations for London social stock): {joined.count():,}')

# How many certs have at least one recommendation?
certs_with_recs = joined.select('certificate_number').distinct().count()
total_certs = certs.count()
pct = (certs_with_recs / total_certs * 100) if total_certs > 0 else 0.0
print(f'Certs with at least one recommendation: {certs_with_recs:,} ({pct:.1f}%)')

Joined rows (recommendations for London social stock): 1,176,679
Certs with at least one recommendation: 422,577 (85.9%)


## 2. Parse indicative cost to numeric midpoint

Costs are stored as strings like '£800 - £1,200' or '£4,000 - £6,000'. We extract the midpoint for aggregation.

In [4]:
# The rec dataset is ROW-per-recommendation (not per-property) — one property can have many rec rows.
total_rec_rows = joined.count()
print(f'Total recommendation rows (joined to London social rented certs): {total_rec_rows:,}')
print('Top 20 raw indicative_cost strings — these are RdSAP banded ranges, need parsing to numeric:')
print()
display(joined.groupBy('indicative_cost').count().orderBy('count', ascending=False).limit(20).toPandas().style.format(thousands=","))

Total recommendation rows (joined to London social rented certs): 1,176,679
Top 20 raw indicative_cost strings — these are RdSAP banded ranges, need parsing to numeric:



,indicative_cost,count
0,"£4,000 - £6,000","152,058"
1,"£4,000 - £14,000","136,058"
2,"£800 - £1,200","74,214"
3,"£500 - £1,500","59,596"
4,"£3,500 - £5,500","50,208"
5,"£3,300 - £6,500","49,360"
6,"£2,200 - £3,000","44,602"
7,£15 - £30,"28,232"
8,£100 - £350,"25,972"
9,£15,"25,672"


In [5]:
def cost_midpoint(s):
    """Parse '£800 - £1,200' → 1000.0"""
    if s is None: return None
    import re
    nums = re.findall(r'[\d]+', s.replace(',', ''))
    if len(nums) == 2:
        return (float(nums[0]) + float(nums[1])) / 2
    elif len(nums) == 1:
        return float(nums[0])
    return None

def cost_low(s):
    """Parse '£800 - £1,200' → 800.0 (lower bound)"""
    if s is None: return None
    import re
    nums = re.findall(r'[\d]+', s.replace(',', ''))
    if len(nums) >= 1:
        return float(nums[0])
    return None

def cost_high(s):
    """Parse '£800 - £1,200' → 1200.0 (upper bound)"""
    if s is None: return None
    import re
    nums = re.findall(r'[\d]+', s.replace(',', ''))
    if len(nums) == 2:
        return float(nums[1])
    elif len(nums) == 1:
        return float(nums[0])
    return None

cost_mid_udf  = udf(cost_midpoint, FloatType())
cost_low_udf  = udf(cost_low,      FloatType())
cost_high_udf = udf(cost_high,     FloatType())

joined = (
    joined
    .withColumn('cost_midpoint', cost_mid_udf(col('indicative_cost')))
    .withColumn('cost_low',      cost_low_udf(col('indicative_cost')))
    .withColumn('cost_high',     cost_high_udf(col('indicative_cost')))
)

print('Sample with parsed costs (low / mid / high):')
display(joined.select('indicative_cost', 'cost_low', 'cost_midpoint', 'cost_high').distinct() \
      .orderBy('cost_midpoint').limit(15).toPandas().style.format(thousands=","))

Sample with parsed costs (low / mid / high):


,indicative_cost,cost_low,cost_midpoint,cost_high
0,£0,0.000000,0.000000,0.000000
1,5,5.000000,5.000000,5.000000
2,£5,5.000000,5.000000,5.000000
3,Â£5,5.000000,5.000000,5.000000
4,£10,10.000000,10.000000,10.000000
5,$10,10.000000,10.000000,10.000000
6,10,10.000000,10.000000,10.000000
7,￡10,10.000000,10.000000,10.000000
8,Â£10,10.000000,10.000000,10.000000
9,15,15.000000,15.000000,15.000000


## 3. Most common improvement types across London social stock

In [6]:
print('=== Most commonly recommended improvements (London social rented) ===')
display(joined.groupBy('improvement_type') \
    .agg(
        count('*').alias('times_recommended'),
        spark_round(avg('cost_midpoint'), 0).alias('avg_indicative_cost_£'),
    ) \
    .orderBy(desc('times_recommended')) \
    .limit(20).toPandas().style.format(thousands=","))

=== Most commonly recommended improvements (London social rented) ===


,improvement_type,times_recommended,avg_indicative_cost_£
0,50 mm internal or external wall insulation,"175,883","9,581.000000"
1,Low energy lighting for all fixed outlets,"168,777",29.000000
2,"Solar photovoltaic panels, 2.5 kWp","116,510","6,867.000000"
3,Solar water heating,"103,024","5,096.000000"
4,Floor insulation (solid floor),"81,629","5,401.000000"
5,Cavity wall insulation,"77,987","1,059.000000"
6,Replace boiler with new condensing boiler,"55,471","2,645.000000"
7,Floor insulation (suspended floor),"53,708","1,874.000000"
8,Replace single glazed windows with low-E double glazed windows,"43,034","5,186.000000"
9,Increase loft insulation to 270 mm,"32,475",298.000000


## 4. Most expensive improvement types

Cost matters for retrofit planning — some improvements are cheap and high-impact, others are expensive and unavoidable for old stock.

In [7]:
print('=== Most expensive improvements (avg indicative cost) ===')
display(joined.groupBy('improvement_type') \
    .agg(
        count('*').alias('times_recommended'),
        spark_round(avg('cost_midpoint'), 0).alias('avg_cost_£'),
    ) \
    .filter(col('times_recommended') > 100) \
    .orderBy(desc('avg_cost_£')) \
    .limit(15).toPandas().style.format(thousands=","))

=== Most expensive improvements (avg indicative cost) ===


,improvement_type,times_recommended,avg_cost_£
0,50 mm internal or external wall insulation,"175,883","9,581.000000"
1,"Solar photovoltaic panels, 2.5 kWp","116,510","6,867.000000"
2,Floor insulation (solid floor),"81,629","5,401.000000"
3,Change room heaters to condensing boiler,621,"5,291.000000"
4,Change heating to gas condensing boiler,"3,712","5,281.000000"
5,Replace single glazed windows with low-E double glazed windows,"43,034","5,186.000000"
6,Solar water heating,"103,024","5,096.000000"
7,Replace single glazed windows with low-E double glazing,"19,266","4,900.000000"
8,Wind turbine,"3,333","4,167.000000"
9,Add PV battery,163,"2,750.000000"


## 5. Borough-level retrofit cost analysis

Which boroughs face the highest estimated total and average retrofit costs?

In [8]:
borough_costs = (
    joined
    .groupBy('borough')
    .agg(
        count('certificate_number').alias('total_recommendations'),
        spark_round(avg('cost_midpoint'), 0).alias('avg_cost_per_recommendation_£'),
        spark_round(spark_sum('cost_midpoint') / 1_000_000, 2).alias('total_indicative_cost_£m'),
    )
    .orderBy(desc('total_indicative_cost_£m'))
)

print('=== Borough retrofit cost estimates ===')
display(borough_costs.limit(33).toPandas().style.format(thousands=","))

borough_costs.write.mode('overwrite').parquet(f'{GOLD}/borough_retrofit_costs')
print('Saved borough_retrofit_costs')

=== Borough retrofit cost estimates ===
Saved borough_retrofit_costs


,borough,total_recommendations,avg_cost_per_recommendation_£,total_indicative_cost_£m
0,Lambeth,"74,194","4,018.000000",298.090000
1,Lewisham,"58,121","4,148.000000",241.060000
2,Hackney,"51,663","3,947.000000",203.940000
3,Haringey,"44,348","3,938.000000",174.660000
4,Hammersmith and Fulham,"43,865","3,964.000000",173.900000
5,Enfield,"51,076","3,376.000000",172.410000
6,Islington,"46,876","3,667.000000",171.890000
7,Newham,"47,459","3,606.000000",171.130000
8,Croydon,"46,188","3,670.000000",169.510000
9,Barking and Dagenham,"43,931","3,578.000000",157.200000


## 6. Top improvement types in our priority boroughs

Focus on the top 5 boroughs from the priority ranking. What do they actually need to fix?

In [9]:
priority_boroughs = [
    'Barking and Dagenham', 'Haringey', 'Lambeth',
    'Hammersmith and Fulham', 'Enfield'
]

priority_recs = (
    joined
    .filter(col('borough').isin(priority_boroughs))
    .groupBy('borough', 'improvement_type')
    .agg(
        count('*').alias('times_recommended'),
        spark_round(avg('cost_midpoint'), 0).alias('avg_cost_£')
    )
)

# Top 5 improvements per borough
w = Window.partitionBy('borough').orderBy(desc('times_recommended'))
top_per_borough = priority_recs \
    .withColumn('rnk', rank().over(w)) \
    .filter(col('rnk') <= 5) \
    .orderBy('borough', 'rnk')

print('=== Top 5 recommended improvements per priority borough ===')
display(top_per_borough.limit(50).toPandas().style.format(thousands=","))

=== Top 5 recommended improvements per priority borough ===


,borough,improvement_type,times_recommended,avg_cost_£,rnk
0,Barking and Dagenham,Low energy lighting for all fixed outlets,"6,698",29.000000,1
1,Barking and Dagenham,"Solar photovoltaic panels, 2.5 kWp","6,256","7,348.000000",2
2,Barking and Dagenham,Solar water heating,"5,378","5,176.000000",3
3,Barking and Dagenham,50 mm internal or external wall insulation,"4,590","9,673.000000",4
4,Barking and Dagenham,Floor insulation (suspended floor),"3,617","1,922.000000",5
5,Enfield,Low energy lighting for all fixed outlets,"7,153",35.000000,1
6,Enfield,50 mm internal or external wall insulation,"6,192","9,353.000000",2
7,Enfield,"Solar photovoltaic panels, 2.5 kWp","5,873","6,835.000000",3
8,Enfield,Solar water heating,"5,176","5,063.000000",4
9,Enfield,Cavity wall insulation,"3,148","1,057.000000",5


## 7. Wall insulation — the key indicator for old stock

Solid wall insulation is the most expensive and most critical improvement for pre-1950 stock. How prevalent is it in our priority boroughs?

In [10]:
wall_insulation_keywords = ['wall insulation', 'solid wall', 'cavity wall']

# Check how improvement_type is worded
display(joined.filter(
    col('improvement_type').rlike('(?i)wall')
).groupBy('improvement_type').count().orderBy(desc('count')).limit(10).toPandas().style.format(thousands=","))

,improvement_type,count
0,50 mm internal or external wall insulation,"175,883"
1,Cavity wall insulation,"77,987"
2,Party wall insulation,"5,565"
3,Internal insulation with cavity wall insulation,4


In [11]:
wall_flag = col('improvement_type').rlike('(?i)wall insulation')

wall_by_borough = (
    joined
    .groupBy('borough')
    .agg(
        count('certificate_number').alias('total_recs'),
        spark_sum(when(wall_flag, 1).otherwise(0)).alias('wall_insulation_recs'),
    )
    .withColumn(
        'pct_needing_wall_insulation',
        spark_round(col('wall_insulation_recs') / col('total_recs') * 100, 1)
    )
    .orderBy(desc('pct_needing_wall_insulation'))
)

print('=== % of recommendations including wall insulation by borough ===')
display(wall_by_borough.limit(33).toPandas().style.format(thousands=","))

=== % of recommendations including wall insulation by borough ===


,borough,total_recs,wall_insulation_recs,pct_needing_wall_insulation
0,None,13,4,30.800000
1,Kensington and Chelsea,"27,716","8,489",30.600000
2,Westminster,"32,719","9,811",30.000000
3,Hammersmith and Fulham,"43,865","12,851",29.300000
4,Hackney,"51,663","15,069",29.200000
5,Lambeth,"74,194","21,570",29.100000
6,Islington,"46,876","13,024",27.800000
7,Camden,"37,499","9,864",26.300000
8,Lewisham,"58,121","15,012",25.800000
9,Tower Hamlets,"45,136","11,084",24.600000


## 8. Retrofit cost scenarios — optimistic, central, pessimistic

The indicative costs are RdSAP bands — fixed national ranges, not precise quotes.
Rather than treating the midpoint as a single estimate, we use all three bounds
to produce a cost range per borough:

- **Optimistic**: every improvement comes in at the lower bound of its cost range
- **Central**: midpoints (our existing estimate)
- **Pessimistic**: every improvement comes in at the upper bound

This gives a realistic cost corridor rather than a single figure, which is more
useful for actual budget planning — and more honest about the uncertainty in RdSAP estimates.

In [12]:
borough_scenarios = (
    joined
    .groupBy('borough')
    .agg(
        count('certificate_number').alias('total_recommendations'),
        countDistinct('certificate_number').alias('properties_with_recs'),
        spark_round(spark_sum('cost_low')      / 1_000_000, 2).alias('total_optimistic_£m'),
        spark_round(spark_sum('cost_midpoint') / 1_000_000, 2).alias('total_central_£m'),
        spark_round(spark_sum('cost_high')     / 1_000_000, 2).alias('total_pessimistic_£m'),
        spark_round(spark_sum('cost_low')      / countDistinct('certificate_number'), 0).alias('per_home_optimistic_£'),
        spark_round(spark_sum('cost_midpoint') / countDistinct('certificate_number'), 0).alias('per_home_central_£'),
        spark_round(spark_sum('cost_high')     / countDistinct('certificate_number'), 0).alias('per_home_pessimistic_£'),
        spark_round(
            (spark_sum('cost_high') - spark_sum('cost_low')) / spark_sum('cost_midpoint') * 100, 1
        ).alias('uncertainty_pct'),
    )
    .orderBy(desc('per_home_central_£'))
)

print('=== Borough retrofit cost scenarios ===')
print('Sorted by cost per home (central estimate) — shows which boroughs have most expensive stock to retrofit.')
print('Total cost (£m) shows budget required to retrofit all social rented stock in each borough.')
print()
display(borough_scenarios.select(
    'borough', 'properties_with_recs',
    'per_home_optimistic_£', 'per_home_central_£', 'per_home_pessimistic_£',
    'total_central_£m', 'uncertainty_pct'
).limit(33).toPandas().style.format(thousands=","))

borough_scenarios.write.mode('overwrite').parquet(f'{GOLD}/borough_retrofit_scenarios')
print('Saved borough_retrofit_scenarios')

=== Borough retrofit cost scenarios ===
Sorted by cost per home (central estimate) — shows which boroughs have most expensive stock to retrofit.
Total cost (£m) shows budget required to retrofit all social rented stock in each borough.

Saved borough_retrofit_scenarios


,borough,properties_with_recs,per_home_optimistic_£,per_home_central_£,per_home_pessimistic_£,total_central_£m,uncertainty_pct
0,Barking and Dagenham,"12,155","9,684.000000","12,933.000000","16,182.000000",157.200000,50.200000
1,Haringey,"14,338","8,182.000000","12,181.000000","16,180.000000",174.660000,65.700000
2,Merton,"6,746","8,441.000000","11,794.000000","15,146.000000",79.560000,56.800000
3,Hillingdon,"10,978","9,123.000000","11,789.000000","14,454.000000",129.420000,45.200000
4,Croydon,"15,329","8,070.000000","11,058.000000","14,046.000000",169.510000,54.000000
5,Bromley,"10,201","8,287.000000","11,026.000000","13,766.000000",112.480000,49.700000
6,Hammersmith and Fulham,"15,911","6,915.000000","10,930.000000","14,945.000000",173.900000,73.500000
7,Harrow,"6,017","8,433.000000","10,810.000000","13,188.000000",65.050000,44.000000
8,Redbridge,"6,914","7,705.000000","10,606.000000","13,508.000000",73.330000,54.700000
9,Enfield,"16,303","7,436.000000","10,575.000000","13,715.000000",172.410000,59.400000


In [13]:
priority_boroughs = [
    'Barking and Dagenham', 'Haringey', 'Lambeth',
    'Hammersmith and Fulham', 'Enfield'
]

print('=== Cost scenarios for TOP 5 PRIORITY BOROUGHS ===')
print('Per-home cost shows retrofit difficulty; total cost shows budget required.')
(
    joined
    .filter(col('borough').isin(priority_boroughs))
    .groupBy('borough')
    .agg(
        countDistinct('certificate_number').alias('properties'),
        spark_round(spark_sum('cost_low')      / countDistinct('certificate_number'), 0).alias('per_home_low_£'),
        spark_round(spark_sum('cost_midpoint') / countDistinct('certificate_number'), 0).alias('per_home_mid_£'),
        spark_round(spark_sum('cost_high')     / countDistinct('certificate_number'), 0).alias('per_home_high_£'),
        spark_round(spark_sum('cost_low')      / 1_000_000, 2).alias('total_low_£m'),
        spark_round(spark_sum('cost_midpoint') / 1_000_000, 2).alias('total_mid_£m'),
        spark_round(spark_sum('cost_high')     / 1_000_000, 2).alias('total_high_£m'),
    )
    .orderBy(desc('per_home_mid_£'))
).show(truncate=False)

print()
print('=== Most expensive improvement types — scenario range ===')
display(
    joined
    .groupBy('improvement_type')
    .agg(
        count('*').alias('times_recommended'),
        spark_round(avg('cost_low'),      0).alias('avg_low_£'),
        spark_round(avg('cost_midpoint'), 0).alias('avg_mid_£'),
        spark_round(avg('cost_high'),     0).alias('avg_high_£'),
    )
    .filter(col('times_recommended') > 100)
    .orderBy(desc('avg_mid_£'))
    .limit(20).toPandas().style.format(thousands=","))

=== Cost scenarios for TOP 5 PRIORITY BOROUGHS ===
Per-home cost shows retrofit difficulty; total cost shows budget required.
+----------------------+----------+--------------+--------------+---------------+------------+------------+-------------+
|borough               |properties|per_home_low_£|per_home_mid_£|per_home_high_£|total_low_£m|total_mid_£m|total_high_£m|
+----------------------+----------+--------------+--------------+---------------+------------+------------+-------------+
|Barking and Dagenham  |12155     |9684.0        |12933.0       |16182.0        |117.71      |157.2       |196.69       |
|Haringey              |14338     |8182.0        |12181.0       |16180.0        |117.32      |174.66      |231.99       |
|Hammersmith and Fulham|15911     |6915.0        |10930.0       |14945.0        |110.02      |173.9       |237.79       |
|Enfield               |16303     |7436.0        |10575.0       |13715.0        |121.23      |172.41      |223.59       |
|Lambeth            

,improvement_type,times_recommended,avg_low_£,avg_mid_£,avg_high_£
0,50 mm internal or external wall insulation,"175,883","5,436.000000","9,581.000000","13,727.000000"
1,"Solar photovoltaic panels, 2.5 kWp","116,510","5,573.000000","6,867.000000","8,161.000000"
2,Floor insulation (solid floor),"81,629","4,374.000000","5,401.000000","6,429.000000"
3,Change room heaters to condensing boiler,621,"3,475.000000","5,291.000000","7,106.000000"
4,Change heating to gas condensing boiler,"3,712","3,299.000000","5,281.000000","7,263.000000"
5,Replace single glazed windows with low-E double glazed windows,"43,034","3,916.000000","5,186.000000","6,455.000000"
6,Solar water heating,"103,024","4,187.000000","5,096.000000","6,005.000000"
7,Replace single glazed windows with low-E double glazing,"19,266","3,300.000000","4,900.000000","6,500.000000"
8,Wind turbine,"3,333","2,611.000000","4,167.000000","5,723.000000"
9,Add PV battery,163,500.000000,"2,750.000000","5,000.000000"


## 9. Cavity wall vs solid wall insulation — retrofit difficulty by borough

**This is the key cost differentiator.**

- **Improvement ID 6** = cavity wall insulation (~£1,500). Applies to properties built after ~1920 with a gap between inner and outer brick layers. Relatively quick and cheap.
- **Improvement ID 7** = internal or external solid wall insulation (~£8,000–£25,000). Applies to pre-1920 solid-brick and pre-1940s properties. The most expensive single retrofit measure.

A borough where most wall insulation recommendations are solid wall (ID 7) faces fundamentally more expensive retrofit than one where most are cavity (ID 6), even if both have the same % of stock below EPC C. This is a more precise retrofit difficulty signal than % pre-1950 stock, and is saved to gold for use in the clustering analysis.

In [14]:
# improvement_item IDs (RdSAP schema):
#   6 = cavity wall insulation (cheap, ~£1,500)
#   7 = solid wall insulation (expensive, £8k–£25k) — the dominant retrofit cost driver
from pyspark.sql.functions import col, count, countDistinct, when, round, desc, avg, sum as spark_sum

wall_by_borough = (
    joined
    .groupBy('borough')
    .agg(
        count('certificate_number').alias('total_recs'),
        spark_sum(when(col('improvement_id').cast('int') == 6, 1).otherwise(0)).alias('cavity_wall_recs'),
        spark_sum(when(col('improvement_id').cast('int') == 7, 1).otherwise(0)).alias('solid_wall_recs'),
    )
    .withColumn('total_wall_recs', col('cavity_wall_recs') + col('solid_wall_recs'))
    .withColumn('pct_solid_wall',
        spark_round(col('solid_wall_recs') / col('total_wall_recs') * 100, 1)
    )
    .withColumn('pct_cavity_wall',
        spark_round(col('cavity_wall_recs') / col('total_wall_recs') * 100, 1)
    )
    .filter(col('total_wall_recs') > 0)
    .orderBy(desc('pct_solid_wall'))
)

print('=== Cavity vs solid wall insulation recommendations by borough ===')
print('Higher % solid wall = more expensive and difficult retrofit stock')
display(wall_by_borough.select(
    'borough', 'total_wall_recs', 'cavity_wall_recs', 'solid_wall_recs',
    'pct_cavity_wall', 'pct_solid_wall'
).limit(33).toPandas().style.format(thousands=","))

# Save to gold — used in 07_clustering_analysis.ipynb as retrofit difficulty metric
wall_by_borough.write.mode('overwrite').parquet(f'{GOLD}/borough_wall_type')
print('Saved borough_wall_type to gold')

=== Cavity vs solid wall insulation recommendations by borough ===
Higher % solid wall = more expensive and difficult retrofit stock
Saved borough_wall_type to gold


,borough,total_wall_recs,cavity_wall_recs,solid_wall_recs,pct_cavity_wall,pct_solid_wall
0,None,4,0,4,0.000000,100.000000
1,Wandsworth,"9,081","1,688","7,393",18.600000,81.400000
2,Haringey,"9,824","1,924","7,900",19.600000,80.400000
3,Hackney,"14,920","3,170","11,750",21.200000,78.800000
4,Hammersmith and Fulham,"12,723","2,721","10,002",21.400000,78.600000
5,Camden,"9,816","2,099","7,717",21.400000,78.600000
6,Lambeth,"21,400","4,752","16,648",22.200000,77.800000
7,Westminster,"9,657","2,212","7,445",22.900000,77.100000
8,Southwark,"9,590","2,267","7,323",23.600000,76.400000
9,Kensington and Chelsea,"8,324","2,000","6,324",24.000000,76.000000


## 10. Cost by improvement category

The 35+ individual improvement types group into 6 meaningful categories. Viewing cost at category level is cleaner for reporting and Warm Homes Fund bids — it tells you whether a borough's retrofit cost is driven by expensive structural works (wall insulation) or cheaper mechanical upgrades (heating controls, hot water).

Categories:
- **Insulation** — wall, loft, floor, roof, party wall
- **Heating system** — boiler replacement, storage heaters, heat pumps, warm air units
- **Heating controls** — thermostats, zone controls, programmer upgrades
- **Glazing & draughts** — double glazing, secondary glazing, draught proofing, doors
- **Hot water** — cylinder insulation, cylinder thermostats
- **Renewables** — solar PV, solar thermal, wind turbine, heat recovery

In [15]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

def categorise(improvement):
    if improvement is None:
        return 'Other'
    t = improvement.lower()
    if any(x in t for x in ['wall insulation', 'loft insulation', 'floor insulation',
                              'roof insulation', 'room-in-roof', 'flat roof', 'party wall']):
        return 'Insulation'
    if any(x in t for x in ['boiler', 'heating unit', 'warm air', 'storage heater',
                              'heat pump', 'biomass', 'wood pellet', 'room heater']):
        return 'Heating system'
    if any(x in t for x in ['heating control', 'zone control', 'thermostat', 'programmer',
                              'time and temperature']):
        return 'Heating controls'
    if any(x in t for x in ['glazing', 'double glaz', 'secondary glaz', 'draught', 'door']):
        return 'Glazing & draughts'
    if any(x in t for x in ['hot water cylinder', 'cylinder insulation', 'cylinder jacket',
                              'cylinder thermostat', 'immersion']):
        return 'Hot water cylinder'
    if any(x in t for x in ['solar', 'photovoltaic', 'wind turbine', 'heat recovery']):
        return 'Renewables'
    if 'lighting' in t:
        return 'Lighting'
    return 'Other'

categorise_udf = udf(categorise, StringType())
joined = joined.withColumn('category', categorise_udf(col('improvement_type')))

print('=== Retrofit cost by category — London social rented stock ===')
(
    joined
    .groupBy('category')
    .agg(
        count('*').alias('recommendations'),
        spark_round(avg('cost_midpoint'), 0).alias('avg_cost_per_rec_£'),
        spark_round(spark_sum('cost_midpoint') / 1_000_000, 1).alias('total_cost_£m'),
        spark_round(spark_sum('cost_low')      / 1_000_000, 1).alias('total_low_£m'),
        spark_round(spark_sum('cost_high')     / 1_000_000, 1).alias('total_high_£m'),
    )
    .orderBy(desc('total_cost_£m'))
).show(truncate=False)

print('=== Cost by category — TOP 5 PRIORITY BOROUGHS ===')
priority_boroughs = ['Barking and Dagenham', 'Haringey', 'Lambeth', 'Hammersmith and Fulham', 'Enfield']
display(
    joined
    .filter(col('borough').isin(priority_boroughs))
    .groupBy('borough', 'category')
    .agg(
        count('*').alias('recs'),
        spark_round(spark_sum('cost_midpoint') / 1_000_000, 2).alias('cost_£m'),
    )
    .orderBy('borough', desc('cost_£m'))
    .limit(50).toPandas().style.format(thousands=","))

=== Retrofit cost by category — London social rented stock ===
+------------------+---------------+------------------+-------------+------------+-------------+
|category          |recommendations|avg_cost_per_rec_£|total_cost_£m|total_low_£m|total_high_£m|
+------------------+---------------+------------------+-------------+------------+-------------+
|Insulation        |495917         |4842.0            |2401.3       |1506.5      |3296.1       |
|Renewables        |231526         |5810.0            |1345.3       |1094.5      |1596.0       |
|Glazing & draughts|114439         |3078.0            |352.2        |262.5       |442.0        |
|Heating system    |92125          |2247.0            |207.0        |171.2       |242.8        |
|Heating controls  |37849          |376.0             |14.2         |12.3        |16.2         |
|Lighting          |168777         |29.0              |4.8          |4.8         |4.9          |
|Hot water cylinder|35840          |24.0              |0.8      

,borough,category,recs,cost_£m
0,Barking and Dagenham,Renewables,"12,274",75.370000
1,Barking and Dagenham,Insulation,"16,738",68.080000
2,Barking and Dagenham,Heating system,"4,259",9.360000
3,Barking and Dagenham,Glazing & draughts,"1,939",3.650000
4,Barking and Dagenham,Heating controls,"1,277",0.500000
5,Barking and Dagenham,Lighting,"6,698",0.200000
6,Barking and Dagenham,Other,13,0.040000
7,Barking and Dagenham,Hot water cylinder,733,0.020000
8,Enfield,Insulation,"20,278",86.510000
9,Enfield,Renewables,"11,630",67.170000


## 11. Quick wins vs major works

Not all outstanding retrofit work is equally difficult or expensive. Splitting by cost tier shows whether a borough still has cheap, quick measures available (loft insulation, cylinder jackets, lighting) or whether the remaining work is almost entirely expensive structural intervention (solid wall insulation, full glazing replacement, boiler replacement).

Thresholds:
- **Quick win**: indicative cost midpoint ≤ £1,000
- **Major works**: indicative cost midpoint > £1,000

In [16]:
joined_tiered = joined.withColumn(
    'work_tier',
    when(col('cost_midpoint') <= 1000, 'Quick win').otherwise('Major works')
)

print('=== Quick wins vs major works — London-wide ===')
(
    joined_tiered
    .groupBy('work_tier')
    .agg(
        count('*').alias('recommendations'),
        spark_round(avg('cost_midpoint'), 0).alias('avg_cost_£'),
        spark_round(spark_sum('cost_midpoint') / 1_000_000, 1).alias('total_cost_£m'),
    )
).show(truncate=False)

print('=== Quick wins vs major works by borough (% of recommendations) ===')
display(
    joined_tiered
    .groupBy('borough')
    .agg(
        count('*').alias('total_recs'),
        spark_round(
            spark_sum(when(col('work_tier') == 'Quick win', 1).otherwise(0)) / count('*') * 100, 1
        ).alias('pct_quick_wins'),
        spark_round(
            spark_sum(when(col('work_tier') == 'Major works', 1).otherwise(0)) / count('*') * 100, 1
        ).alias('pct_major_works'),
        spark_round(spark_sum('cost_midpoint') / countDistinct('certificate_number'), 0).alias('cost_per_home_£'),
    )
    .orderBy(desc('pct_major_works'))
    .limit(33).toPandas().style.format(thousands=","))

print('=== Priority boroughs — quick win vs major works breakdown ===')
display(
    joined_tiered
    .filter(col('borough').isin(priority_boroughs))
    .groupBy('borough', 'work_tier', 'improvement_type')
    .agg(
        count('*').alias('recs'),
        spark_round(avg('cost_midpoint'), 0).alias('avg_cost_£'),
    )
    .filter(col('work_tier') == 'Quick win')
    .orderBy('borough', desc('recs'))
    .limit(40).toPandas().style.format(thousands=","))

=== Quick wins vs major works — London-wide ===
+-----------+---------------+----------+-------------+
|work_tier  |recommendations|avg_cost_£|total_cost_£m|
+-----------+---------------+----------+-------------+
|Quick win  |473267         |405.0     |191.8        |
|Major works|703412         |5878.0    |4134.5       |
+-----------+---------------+----------+-------------+

=== Quick wins vs major works by borough (% of recommendations) ===
=== Priority boroughs — quick win vs major works breakdown ===


,borough,total_recs,pct_quick_wins,pct_major_works,cost_per_home_£
0,None,13,30.800000,69.200000,"8,549.000000"
1,Harrow,"16,969",34.500000,65.500000,"10,810.000000"
2,Hillingdon,"34,270",34.500000,65.500000,"11,789.000000"
3,Lewisham,"58,121",34.600000,65.400000,"10,533.000000"
4,Merton,"21,544",35.600000,64.400000,"11,794.000000"
5,Lambeth,"74,194",36.600000,63.400000,"10,188.000000"
6,Waltham Forest,"36,630",36.700000,63.300000,"10,482.000000"
7,Bromley,"29,561",36.900000,63.100000,"11,026.000000"
8,Kingston upon Thames,"16,730",37.200000,62.800000,"10,004.000000"
9,Westminster,"32,719",37.400000,62.600000,"9,839.000000"


,borough,work_tier,improvement_type,recs,avg_cost_£
0,Barking and Dagenham,Quick win,Low energy lighting for all fixed outlets,"6,698",29.000000
1,Barking and Dagenham,Quick win,Floor insulation (suspended floor),"2,386","1,000.000000"
2,Barking and Dagenham,Quick win,Cavity wall insulation,"2,256","1,000.000000"
3,Barking and Dagenham,Quick win,Floor insulation,"1,546","1,000.000000"
4,Barking and Dagenham,Quick win,Upgrade heating controls,"1,147",399.000000
5,Barking and Dagenham,Quick win,Increase loft insulation to 270 mm,"1,047",243.000000
6,Barking and Dagenham,Quick win,Flue gas heat recovery in conjunction with boiler,639,767.000000
7,Barking and Dagenham,Quick win,Increase hot water cylinder insulation,386,23.000000
8,Barking and Dagenham,Quick win,Draught proof single-glazed windows,277,106.000000
9,Barking and Dagenham,Quick win,Add additional 80 mm jacket to hot water cylinder,269,23.000000


## 12. Priority order analysis — what do assessors recommend first?

`improvement_item` is the assessor's priority rank within each certificate: 1 = single most impactful measure for that property. Analysing which improvement types most often appear at priority 1 tells you what assessors consistently identify as the biggest lever — more actionable than knowing something is merely common.

In [17]:
print('=== Improvements most frequently ranked priority 1 (London social rented) ===')
(
    joined
    .filter(col('priority') == 1)
    .groupBy('improvement_type', 'category')
    .agg(
        count('*').alias('times_top_priority'),
        spark_round(avg('cost_midpoint'), 0).alias('avg_cost_£'),
    )
    .orderBy(desc('times_top_priority'))
    .limit(15)
).show(truncate=False)

print('=== Priority 1 improvement by borough — top 5 priority boroughs ===')
from pyspark.sql.window import Window
from pyspark.sql.functions import rank as spark_rank

w = Window.partitionBy('borough').orderBy(desc('times_top'))
display(
    joined
    .filter((col('priority') == 1) & col('borough').isin(priority_boroughs))
    .groupBy('borough', 'improvement_type', 'category')
    .agg(count('*').alias('times_top'))
    .withColumn('rnk', spark_rank().over(w))
    .filter(col('rnk') <= 3)
    .orderBy('borough', 'rnk')
    .limit(30).toPandas().style.format(thousands=","))

print('=== How priority order affects cost — do expensive works come first? ===')
(
    joined
    .groupBy('priority')
    .agg(
        count('*').alias('recommendations'),
        spark_round(avg('cost_midpoint'), 0).alias('avg_cost_£'),
    )
    .filter(col('priority') <= 10)
    .orderBy('priority')
).show(truncate=False)

=== Improvements most frequently ranked priority 1 (London social rented) ===
+-------------------------------------------------+------------------+------------------+----------+
|improvement_type                                 |category          |times_top_priority|avg_cost_£|
+-------------------------------------------------+------------------+------------------+----------+
|50 mm internal or external wall insulation       |Insulation        |143901            |9571.0    |
|Cavity wall insulation                           |Insulation        |62375             |1058.0    |
|Low energy lighting for all fixed outlets        |Lighting          |38974             |27.0      |
|Floor insulation (solid floor)                   |Insulation        |34919             |5411.0    |
|Increase loft insulation to 270 mm               |Insulation        |32475             |298.0     |
|Flat roof or sloping ceiling insulation          |Insulation        |26103             |1196.0    |
|Floor insula

,borough,improvement_type,category,times_top,rnk
0,Barking and Dagenham,50 mm internal or external wall insulation,Insulation,"3,683",1
1,Barking and Dagenham,Cavity wall insulation,Insulation,"2,465",2
2,Barking and Dagenham,Increase loft insulation to 270 mm,Insulation,"1,128",3
3,Enfield,50 mm internal or external wall insulation,Insulation,"4,194",1
4,Enfield,Cavity wall insulation,Insulation,"2,432",2
5,Enfield,Increase loft insulation to 270 mm,Insulation,"1,902",3
6,Hammersmith and Fulham,50 mm internal or external wall insulation,Insulation,"8,518",1
7,Hammersmith and Fulham,Cavity wall insulation,Insulation,"2,286",2
8,Hammersmith and Fulham,Low energy lighting for all fixed outlets,Lighting,"1,043",3
9,Haringey,50 mm internal or external wall insulation,Insulation,"6,202",1


## The cost of NOT retrofitting

The retrofit cost scenarios in this notebook represent the cost of action. It is worth framing these against the cost of inaction.

**MEES compliance risk:** Any property that fails to reach band C by the 2030 deadline cannot legally be let to new tenants. 
For a housing association, a void property that cannot be re-let is lost rental income plus ongoing maintenance liability. 
At average London social rents (~£120/week), a property sitting void for 6 months while awaiting retrofit costs ~£3,000 in lost income alone — 
often more than the cheaper improvement measures (loft insulation, cylinder jacket, heating controls) that might be all that’s needed to cross the band C threshold.

**Sequencing matters:** Not all properties need expensive structural works to reach band C. 
The quick wins vs major works analysis (section 11) shows what proportion of each borough’s outstanding work is under £1,000. 
Properties close to the band C threshold may only need one or two cheap measures — identifying these first maximises compliance at minimum cost.

**Warm Homes Fund:** The grant covers up to 100% of eligible retrofit costs for the worst-performing properties with the most deprived tenants. 
The cost scenarios here represent the gross cost before grant — the net cost to the association could be substantially lower for priority boroughs.

## 13. Summary

Key findings from the full recommendations analysis:

In [18]:
print('=== RETROFIT COST SUMMARY ===')
print()

print('Top 5 most common improvements London-wide:')
joined.groupBy('improvement_type') \
    .agg(
        count('*').alias('n'),
        spark_round(avg('cost_midpoint'), 0).alias('avg_cost_£')
    ) \
    .orderBy(desc('n')).limit(5).show(truncate=False)

print('Priority boroughs — avg cost per recommendation:')
joined.filter(col('borough').isin(priority_boroughs)) \
    .groupBy('borough') \
    .agg(spark_round(avg('cost_midpoint'), 0).alias('avg_cost_per_rec_£')) \
    .orderBy(desc('avg_cost_per_rec_£')).show(truncate=False)

=== RETROFIT COST SUMMARY ===

Top 5 most common improvements London-wide:
+------------------------------------------+------+----------+
|improvement_type                          |n     |avg_cost_£|
+------------------------------------------+------+----------+
|50 mm internal or external wall insulation|175883|9581.0    |
|Low energy lighting for all fixed outlets |168777|29.0      |
|Solar photovoltaic panels, 2.5 kWp        |116510|6867.0    |
|Solar water heating                       |103024|5096.0    |
|Floor insulation (solid floor)            |81629 |5401.0    |
+------------------------------------------+------+----------+

Priority boroughs — avg cost per recommendation:
+----------------------+------------------+
|borough               |avg_cost_per_rec_£|
+----------------------+------------------+
|Lambeth               |4018.0            |
|Hammersmith and Fulham|3964.0            |
|Haringey              |3938.0            |
|Barking and Dagenham  |3578.0            |


In [19]:
import os
os.makedirs('../outputs', exist_ok=True)

borough_scenarios.toPandas().to_csv('../outputs/borough_retrofit_scenarios.csv', index=False)
print('Saved ../outputs/borough_retrofit_scenarios.csv')

wall_by_borough.toPandas().to_csv('../outputs/borough_wall_type.csv', index=False)
print('Saved ../outputs/borough_wall_type.csv')

Saved outputs/borough_retrofit_scenarios.csv
Saved outputs/borough_wall_type.csv
